# Probabilistic College Basketball Ratings
### A ridge prior is worth three weeks of data — and the posterior knows what it doesn't know

---

Most college basketball rating systems produce a number.  
This one produces a **distribution**.

The difference matters in two places:

- **Early season** — when you have 5 games per team instead of 30, a point estimate is a guess. A posterior is honest about that.
- **Matchup prediction** — knowing a team scores 112 pts/100 tells you something. Knowing they score $112 \pm 3$ vs an opponent who allows $108 \pm 4$ tells you a lot more.

We build two models — a KenPom-style fixed-point system (Model 1) and a Bayesian ridge regression (Model 2) — validate them against each other using a strict **one-step-ahead** protocol, and show:

1. The ridge prior makes the model well-posed from game 1, worth roughly **three weeks of additional data** in early-season forecast accuracy.
2. The posterior predictive intervals are **empirically well-calibrated** — a 90% PI contains about 90% of actual outcomes.
3. The small systematic bias that remains is a clean signature of **within-season team improvement**, pointing at the natural next extension.

## Setup

In [1]:
import sys
from pathlib import Path
from datetime import datetime, timedelta
from collections import defaultdict

repo = Path("../").resolve()
if str(repo) not in sys.path:
    sys.path.insert(0, str(repo))

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from scipy.stats import gaussian_kde

from models.data import open_db, load_season_games, GameRow
from models.model1 import Model1
from models.model2 import Model2
from models.eval import temporal_split

# ── shared plot style ────────────────────────────────────────────────────────
BG, GRID, SPINE, TICK = "#0f0f1a", "#1a1a3a", "#333355", "#aaaacc"
C1, C2 = "#00bfff", "#ff6b35"

def dark_ax(ax, fig=None):
    ax.set_facecolor(BG)
    if fig: fig.patch.set_facecolor(BG)
    for sp in ax.spines.values(): sp.set_color(SPINE)
    ax.tick_params(colors=TICK, labelsize=9)
    ax.grid(color=GRID, linewidth=0.5, linestyle="--")
    ax.xaxis.label.set_color(TICK)
    ax.yaxis.label.set_color(TICK)

SEASON = 2026
conn   = open_db()
cur    = conn.cursor()

cur.execute("SELECT TeamID, School FROM teams_years WHERE year = ?", (SEASON,))
team_names = dict(cur.fetchall())

def parse_date(s):
    for fmt in ("%b %d, %Y", "%b %d,%Y"):
        try: return datetime.strptime(s.strip(), fmt)
        except ValueError: pass
    raise ValueError(s)

print("Ready.")

Ready.


---
## 1 · The quantity we're predicting

Every model predicts **offensive efficiency**: points scored per 100 possessions.

$$e_{\text{off},ig} = \frac{\text{pts}_{ig}}{\text{poss}_{ig}} \times 100$$

Each game yields two observations (one per team side). A full season of ~6,300 games gives ~12,600 rows. League average sits around 107–110; elite offenses reach 120+.

In [ ]:
all_rows = load_season_games(conn, SEASON)
e_off    = np.array([r.pts / r.poss * 100.0 for r in all_rows])

fig, ax = plt.subplots(figsize=(10, 4))
dark_ax(ax, fig)
ax.hist(e_off, bins=80, color=C2, alpha=0.85, edgecolor="none")
ax.axvline(e_off.mean(), color="white", linewidth=1.2, linestyle="--",
           label=f"mean = {e_off.mean():.1f}")
ax.set_xlabel("Offensive efficiency  (pts / 100 poss)")
ax.set_ylabel("Game-team observations")
ax.set_title(f"2025-26 season — {len(all_rows):,} observations,  "
             f"σ = {e_off.std():.1f} pts/100",
             color="white", fontweight="bold")
ax.legend(labelcolor="white", facecolor="#111133", edgecolor=SPINE)
plt.tight_layout(); plt.show()
print(f"Mean {e_off.mean():.2f}   Std {e_off.std():.2f}   "
      f"5th pct {np.percentile(e_off,5):.1f}   95th pct {np.percentile(e_off,95):.1f}")

---
## 2 · Model 1 — Fixed-point iteration

Each team has latent scalars $(O_i, D_i, R_i)$ defined by self-consistency. For offense:

$$O_i^{(t+1)} = \frac{\displaystyle\sum_g w_g \cdot e_{\text{off},ig} \cdot \frac{\bar\mu}{D_j^{(t)}} \cdot L(h)^{-1}}{\displaystyle\sum_g w_g}$$

where $w_g = \text{poss}_g / \bar{\text{poss}}$ and $L(h) \in \{1.014, 1.0, 0.986\}$ for home / neutral / away.

Prediction is multiplicative: $\hat{e}_{\text{off}} = O_i \cdot D_j / \bar\mu \cdot L(h)$.

**Limitation:** no regularization. With 350 teams and ~5 games each in week 1, the system is nearly underdetermined.

In [ ]:
m1_full = Model1()
m1_full.fit_rows(all_rows, SEASON)
summary1 = m1_full.point_summary()

top10 = sorted(summary1.items(), key=lambda x: x[1].net_rtg, reverse=True)[:10]
print(f"{'Team':<22} {'AdjO':>6} {'AdjD':>6} {'Net':>7}")
print("-" * 46)
for tid, s in top10:
    print(f"{team_names.get(tid, str(tid)):<22} {s.adj_o:>6.1f} {s.adj_d:>6.1f} {s.net_rtg:>+7.1f}")

---
## 3 · Model 2 — Ridge regression with exact Gaussian posterior

The additive observation model:

$$e_{\text{off},ig} = \mu + o_i - d_j + \eta \cdot h_g + \varepsilon_g, \qquad \varepsilon_g \sim \mathcal{N}\!\left(0,\, \frac{\sigma^2}{p_g}\right)$$

With Gaussian priors $o_i, d_j \sim \mathcal{N}(0, \lambda^{-1})$, the MAP is the ridge solution:

$$\hat\theta = (X^\top W X + \Lambda)^{-1} X^\top W y$$

And the **exact posterior** is:

$$p(\theta \mid \mathcal{D}) = \mathcal{N}\!\left(\hat\theta,\; \hat\sigma^2 (X^\top W X + \Lambda)^{-1}\right)$$

The KenPom-style summary is derived from the posterior:
- $\text{AdjO}_i = \mu + o_i$
- $\text{AdjD}_i = \mu - d_i$  
- $\text{AdjPace}_i = \exp(\gamma_0 + r_i)$

**The key difference from Model 1:** $\Lambda$ regularizes toward the league average. The system is always well-posed, even with one game per team.

In [ ]:
m2_full = Model2()
m2_full.fit_rows(all_rows, SEASON)
summary2 = m2_full.point_summary()

top10_m2 = sorted(summary2.items(), key=lambda x: x[1].net_rtg, reverse=True)[:10]
print(f"{'Team':<22} {'AdjO':>6} {'AdjD':>6} {'Net':>7}")
print("-" * 46)
for tid, s in top10_m2:
    print(f"{team_names.get(tid, str(tid)):<22} {s.adj_o:>6.1f} {s.adj_d:>6.1f} {s.net_rtg:>+7.1f}")

# Posterior uncertainty on net rating for top teams
rng_demo = np.random.default_rng(42)
draws_demo = m2_full.sample_posterior(500, rng_demo)
T = len(m2_full.teams_)
net_draws = np.array([th[1:T+1] + th[T+1:2*T+1] for th in draws_demo])

print("\nPosterior std of net rating (top 10):")
for tid, s in top10_m2:
    pos = list(m2_full.teams_).index(tid)
    std = float(net_draws[:, pos].std())
    print(f"  {team_names.get(tid, str(tid)):<22}  σ = {std:.2f} pts/100")

---
## 4 · One-step-ahead validation

For each week $W$:
1. Fit on **all games strictly before week $W$**
2. Predict games **in week $W$**
3. Record residuals

This is the only protocol that mirrors deployment — the model never sees a game before predicting it.

In [ ]:
cur.execute("""
    SELECT DISTINCT s.GameID, s.Date
    FROM schedules s JOIN boxscores b ON s.GameID = b.GameID
    WHERE s.Year = ? ORDER BY s.GameID
""", (SEASON,))
game_order = cur.fetchall()
game_dt    = {gid: parse_date(ds) for gid, ds in game_order}
all_gids   = [gid for gid, _ in game_order]

row_by_gid = defaultdict(list)
for r in all_rows:
    row_by_gid[r.game_id].append(r)

WEEK_STEP, MIN_TRAIN = 7, 300
season_start = min(game_dt.values())
season_end   = max(game_dt.values())

windows = []
d = season_start
while d <= season_end:
    d_next = d + timedelta(days=WEEK_STEP)
    train  = {g for g in all_gids if game_dt[g] < d}
    pred   = {g for g in all_gids if d <= game_dt[g] < d_next}
    if len(train) >= MIN_TRAIN and pred:
        windows.append((d, train, pred))
    d = d_next

print(f"{len(windows)} prediction windows  ({season_start.strftime('%b %d')} – {season_end.strftime('%b %d, %Y')})")

In [ ]:
records = []  # (date, actual, pred1, pred2)

for win_dt, train_gids, pred_gids in windows:
    t_rows = [r for g in train_gids for r in row_by_gid[g]]
    p_rows = [r for g in pred_gids  for r in row_by_gid[g]]
    m1 = Model1(); m1.fit_rows(t_rows, SEASON)
    m2 = Model2(); m2.fit_rows(t_rows, SEASON)
    actual = np.array([r.pts / r.poss * 100.0 for r in p_rows])
    records.append((win_dt, actual,
                    m1.predict_efficiency(p_rows),
                    m2.predict_efficiency(p_rows)))
    r1 = float(np.sqrt(np.mean((records[-1][2] - actual)**2)))
    r2 = float(np.sqrt(np.mean((records[-1][3] - actual)**2)))
    print(f"  {win_dt.strftime('%b %d')}  train={len(train_gids):>4}  "
          f"pred_games={len(pred_gids):>3}  RMSE1={r1:.2f}  RMSE2={r2:.2f}")

### 4a · The headline result

In [ ]:
dates  = [r[0] for r in records]
rmse1s = [float(np.sqrt(np.mean((r[2]-r[1])**2))) for r in records]
rmse2s = [float(np.sqrt(np.mean((r[3]-r[1])**2))) for r in records]
bias1s = [float(np.mean(r[2]-r[1])) for r in records]
bias2s = [float(np.mean(r[3]-r[1])) for r in records]

all_r1, all_r2, cum1, cum2 = [], [], [], []
for r in records:
    all_r1.extend((r[2]-r[1]).tolist())
    all_r2.extend((r[3]-r[1]).tolist())
    cum1.append(float(np.sqrt(np.mean(np.array(all_r1)**2))))
    cum2.append(float(np.sqrt(np.mean(np.array(all_r2)**2))))

fig, axes = plt.subplots(3, 1, figsize=(12, 10), sharex=True)
fig.patch.set_facecolor(BG)
for ax in axes: dark_ax(ax)

# Panel 1: weekly RMSE
axes[0].plot(dates, rmse1s, color=C1, lw=2.0, label="Model 1  (fixed-point)")
axes[0].plot(dates, rmse2s, color=C2, lw=2.0, label="Model 2  (ridge)")
axes[0].fill_between(dates, rmse1s, rmse2s,
    where=[r2<r1 for r1,r2 in zip(rmse1s,rmse2s)], color=C2, alpha=0.15)
axes[0].fill_between(dates, rmse1s, rmse2s,
    where=[r1<r2 for r1,r2 in zip(rmse1s,rmse2s)], color=C1, alpha=0.15)
axes[0].annotate(
    "Fixed-point underdetermined\n(~700 games, 350 teams)",
    xy=(dates[0], rmse1s[0]), xytext=(dates[2], rmse1s[0]-3.5),
    color="#ccccff", fontsize=8,
    arrowprops=dict(arrowstyle="->", color="#555577", lw=1.0))
axes[0].set_ylabel("Weekly RMSE  (pts/100)")
axes[0].set_title("One-step-ahead forecast quality — 2025-26",
                  color="white", fontsize=11, fontweight="bold")
axes[0].legend(labelcolor="white", facecolor="#111133", edgecolor=SPINE, fontsize=9)

# Panel 2: cumulative RMSE
axes[1].plot(dates, cum1, color=C1, lw=2.5, label="Model 1  cumulative")
axes[1].plot(dates, cum2, color=C2, lw=2.5, label="Model 2  cumulative")
stable = next(i for i,r in enumerate(rmse1s) if r < 15.5)
axes[1].axvline(dates[stable], color="#555577", lw=1.0, linestyle=":")
axes[1].annotate(f"{dates[stable].strftime('%b %d')} — Model 1 stabilizes",
                 xy=(dates[stable], cum1[stable]),
                 xytext=(8, -18), textcoords="offset points",
                 color="#888899", fontsize=8)
axes[1].set_ylabel("Cumulative RMSE  (pts/100)")
axes[1].set_title("Cumulative RMSE — ridge prior worth ~3 weeks of data",
                  color="white", fontsize=11, fontweight="bold")
axes[1].legend(labelcolor="white", facecolor="#111133", edgecolor=SPINE, fontsize=9)

# Panel 3: bias
axes[2].plot(dates, bias1s, color=C1, lw=1.8, label="Model 1")
axes[2].plot(dates, bias2s, color=C2, lw=1.8, label="Model 2")
axes[2].axhline(0, color="#555577", lw=0.8, linestyle=":")
axes[2].set_ylabel("Bias  (pred − actual,  pts/100)")
axes[2].set_title("Weekly bias — systematic underprediction = team improvement signal",
                  color="white", fontsize=11, fontweight="bold")
axes[2].legend(labelcolor="white", facecolor="#111133", edgecolor=SPINE, fontsize=9)
axes[2].xaxis.set_major_formatter(mdates.DateFormatter("%b '%y"))
axes[2].xaxis.set_major_locator(mdates.MonthLocator())

plt.tight_layout(); plt.show()

all_actual = np.concatenate([r[1] for r in records])
all_p1     = np.concatenate([r[2] for r in records])
all_p2     = np.concatenate([r[3] for r in records])
print(f"\nSeason totals  (N={len(all_actual):,} team-game observations)")
for name, p in [("Model 1", all_p1), ("Model 2", all_p2)]:
    res = p - all_actual
    print(f"  {name}  RMSE={np.sqrt(np.mean(res**2)):.3f}  "
          f"MAE={np.mean(np.abs(res)):.3f}  bias={np.mean(res):+.3f}")

---
## 5 · Calibration

A system can be accurate on average and still misrepresent its own uncertainty. The calibration curve tests the posterior directly: at every nominal coverage level $1-\alpha$, does the PI contain that fraction of actuals?

The **predictive distribution** integrates two sources of uncertainty:

$$p(\tilde{e}_{ij} \mid \mathcal{D}) = \underbrace{\int p(\tilde{e}_{ij}\mid\theta)\,p(\theta\mid\mathcal{D})\,d\theta}_{\text{parameter uncertainty}} \;+\; \underbrace{\mathcal{N}(0,\sigma^2_{\text{eff}})}_{\text{irreducible noise}}$$

In [ ]:
train_rows, test_rows = temporal_split(all_rows, 0.80)
m_cal = Model2()
m_cal.fit_rows(train_rows, SEASON)

actual_cal = np.array([r.pts / r.poss * 100.0 for r in test_rows])
draws_cal  = m_cal.sample_posterior(400, np.random.default_rng(42))
preds_cal  = np.array([m_cal._predict_from_theta(th, test_rows) for th in draws_cal])
sigma_cal  = float(np.sqrt(m_cal._sigma2_eff))

levels, emp_cov = np.linspace(0.02, 0.99, 60), []
rng0 = np.random.default_rng(0).standard_normal(100_000)
for lv in levels:
    alpha = (1 - lv) / 2
    z  = float(np.abs(np.quantile(rng0, alpha)))
    lo = np.quantile(preds_cal, alpha,     axis=0) - z * sigma_cal
    hi = np.quantile(preds_cal, 1 - alpha, axis=0) + z * sigma_cal
    emp_cov.append(float(np.mean((actual_cal >= lo) & (actual_cal <= hi))))

fig, ax = plt.subplots(figsize=(7, 7))
dark_ax(ax, fig)
ax.plot([0,1],[0,1], color="#555588", lw=1.5, linestyle="--", label="Perfect calibration")
ax.plot(levels, emp_cov, color=C1, lw=2.5, label="Model 2 (ridge)")
ax.fill_between(levels, levels, emp_cov,
    where=[e>n for e,n in zip(emp_cov,levels)], color="#00ff88", alpha=0.12, label="Underconfident")
ax.fill_between(levels, levels, emp_cov,
    where=[e<n for e,n in zip(emp_cov,levels)], color="#ff4466", alpha=0.12, label="Overconfident")
for lv_mark in [0.50, 0.80, 0.90, 0.95]:
    idx = np.argmin(np.abs(levels - lv_mark))
    ax.annotate(f"{emp_cov[idx]:.0%} actual\n@ {lv_mark:.0%} nominal",
                xy=(lv_mark, emp_cov[idx]), xytext=(lv_mark-0.22, emp_cov[idx]+0.04),
                color="#ccccff", fontsize=7.5,
                arrowprops=dict(arrowstyle="-", color="#555588", lw=0.8))
ax.set_xlim(0,1); ax.set_ylim(0,1)
ax.set_xlabel("Nominal coverage  (1 − α)")
ax.set_ylabel("Empirical coverage")
ax.set_title("Calibration curve — posterior predictive intervals\n"
             "2025-26  ·  80/20 train/test  ·  Model 2 (ridge)",
             color="white", fontsize=11, fontweight="bold")
ax.legend(labelcolor="white", facecolor="#111133", edgecolor=SPINE, fontsize=9)
plt.tight_layout(); plt.show()

---
## 6 · Team uncertainty through the season

Re-fitting weekly gives team net-rating trajectories with posterior $\pm 1\sigma / \pm 2\sigma$ bands. Uncertainty shrinks as games accumulate; by late January the bands are narrow and stable.

In [ ]:
milestones, seen_dt = [], set()
d = season_start
while d <= season_end + timedelta(days=1):
    gids = {g for g in all_gids if game_dt[g] < d}
    if len(gids) >= MIN_TRAIN and d not in seen_dt:
        milestones.append((d - timedelta(days=1), gids)); seen_dt.add(d)
    d += timedelta(days=WEEK_STEP)
milestones.append((season_end, set(all_gids)))

fan = defaultdict(list)
rng_fan = np.random.default_rng(42)
for cutoff, gids in milestones:
    rows_here = [r for g in gids for r in row_by_gid[g]]
    m = Model2(); m.fit_rows(rows_here, SEASON)
    T = len(m.teams_)
    dr = m.sample_posterior(200, rng_fan)
    net = np.array([th[1:T+1] + th[T+1:2*T+1] for th in dr])
    for pos, tid in enumerate(m.teams_):
        fan[int(tid)].append((cutoff, float(net[:,pos].mean()), float(net[:,pos].std())))
    print(f"  {cutoff.strftime('%b %d')}  games={len(gids)}", flush=True)

print("Done.")

In [ ]:
TOP_N = 15
final_net = {tid: pts[-1][1] for tid, pts in fan.items()}
top_tids  = sorted(final_net, key=final_net.get, reverse=True)[:TOP_N]
cmap_t    = plt.get_cmap("tab20")
colors_t  = [cmap_t(i/TOP_N) for i in range(TOP_N)]

fig, ax = plt.subplots(figsize=(14, 7))
dark_ax(ax, fig)

label_entries = []
for tid, color in zip(top_tids, colors_t):
    pts   = fan[tid]
    dts   = [p[0] for p in pts]
    mn    = np.array([p[1] for p in pts])
    sd    = np.array([p[2] for p in pts])
    ax.fill_between(dts, mn-2*sd, mn+2*sd, color=color, alpha=0.10)
    ax.fill_between(dts, mn-sd,   mn+sd,   color=color, alpha=0.28)
    ax.plot(dts, mn, color=color, lw=1.8, zorder=3)
    label_entries.append((team_names.get(tid, str(tid)), color, mn[-1]))

label_entries.sort(key=lambda x: x[2], reverse=True)
x_end  = max(p[0] for pts in fan.values() for p in pts)
x_span = x_end - min(p[0] for pts in fan.values() for p in pts)
ax.set_xlim(min(p[0] for pts in fan.values() for p in pts), x_end + x_span*0.20)

placed = []
for name, color, val in label_entries:
    y = val
    for py in placed:
        if abs(y-py) < 1.1: y = min(py-1.1, y)
    placed.append(y)
    ax.annotate(name, xy=(x_end, val), xytext=(10,0),
                textcoords="offset points", color=color,
                fontsize=8, va="center", fontweight="bold",
                annotation_clip=False)

ax.xaxis.set_major_formatter(mdates.DateFormatter("%b '%y"))
ax.xaxis.set_major_locator(mdates.MonthLocator())
ax.set_ylabel("Net Rating  (AdjO − AdjD,  pts / 100 poss)")
ax.set_title(f"2025-26 Rolling Net Rating — top {TOP_N} teams\n"
             "shading = ±1σ / ±2σ posterior  ·  Model 2 (ridge)",
             color="white", fontsize=12, fontweight="bold")
ax.axhline(0, color="#555577", lw=0.7, linestyle=":")
plt.tight_layout(); plt.show()

---
## 7 · Pre-game posterior predictive distributions

For any matchup the full posterior predictive distribution is available:

$$p(\tilde{e}_{ij} \mid \mathcal{D}) \approx \frac{1}{S}\sum_{s=1}^S \mathcal{N}\!\left(\hat{e}_{ij}^{(s)},\, \sigma^2_{\text{eff}}\right), \qquad \theta^{(s)} \sim p(\theta\mid\mathcal{D})$$

Overlap between two teams' distributions is a direct visual measure of competitiveness. The dotted vertical lines are the actual outcomes — a live calibration check.

In [ ]:
top25 = set(sorted(final_net, key=final_net.get, reverse=True)[:25])
cur.execute("""
    SELECT s.GameID, s.Date, s.TeamID
    FROM schedules s JOIN boxscores b ON s.GameID = b.GameID
    WHERE s.Year = ? ORDER BY s.GameID
""", (SEASON,))
game_meta = {}
for gid, ds, tid in cur.fetchall():
    if gid not in game_meta:
        game_meta[gid] = {"teams": [], "date": parse_date(ds)}
    game_meta[gid]["teams"].append(tid)

elite = [(gid, m) for gid, m in game_meta.items()
         if len(set(m["teams"]) & top25) >= 2
         and m["date"] >= datetime(2026, 1, 15)]
elite.sort(key=lambda x: x[1]["date"])
step = max(1, len(elite)//6)
showcase = [elite[i*step] for i in range(6)][:6]

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
fig.patch.set_facecolor(BG)
SIDE = ["#00bfff", "#ff6b35"]

for ax, (gid, meta) in zip(axes.flatten(), showcase):
    dark_ax(ax)
    t_gids = {g for g in all_gids if game_dt[g] < meta["date"]}
    t_rows = [r for g in t_gids for r in row_by_gid[g]]
    m_gm   = Model2(); m_gm.fit_rows(t_rows, SEASON)
    actual_gm = {r.team_id: r.pts/r.poss*100.0 for r in row_by_gid[gid]}
    dr_gm  = m_gm.sample_posterior(300, np.random.default_rng(42))
    sig_gm = float(np.sqrt(m_gm._sigma2_eff))
    xs = np.linspace(60, 155, 400)

    for idx, r in enumerate(row_by_gid[gid][:2]):
        pp  = np.array([float(m_gm._predict_from_theta(th, [r])[0]) for th in dr_gm])
        pp += np.random.default_rng(42+idx).normal(0, sig_gm, len(pp))
        kde = gaussian_kde(pp, bw_method=0.25)
        dens = kde(xs)
        ax.fill_between(xs, dens, alpha=0.35, color=SIDE[idx])
        ax.plot(xs, dens, color=SIDE[idx], lw=1.8,
                label=team_names.get(r.team_id, str(r.team_id)))
        act = actual_gm.get(r.team_id)
        if act:
            ax.axvline(act, color=SIDE[idx], lw=1.5, linestyle=":")
            ax.annotate(f"{act:.0f}", xy=(act, kde(act)[0]*0.45),
                        xytext=(4,0), textcoords="offset points",
                        color=SIDE[idx], fontsize=8)

    ax.set_title(meta["date"].strftime("%b %d, %Y"), color="white",
                 fontsize=9, fontweight="bold")
    ax.set_xlabel("Offensive efficiency  (pts/100)")
    ax.legend(labelcolor="white", facecolor="#111133",
              edgecolor=SPINE, fontsize=8, loc="upper right")
    ax.text(0.02, 0.97, "dotted = actual", transform=ax.transAxes,
            color="#666688", fontsize=6.5, va="top")

fig.suptitle("Pre-game posterior predictive distributions — top-25 matchups\n"
             "overlap = competitiveness  ·  dotted = actual result  ·  Model 2 (ridge)",
             color="white", fontsize=11, fontweight="bold")
plt.tight_layout(rect=[0,0,1,0.94]); plt.show()

---
## 8 · Summary

| | Model 1 | Model 2 |
|---|---|---|
| **Method** | Fixed-point iteration | Ridge regression |
| **Posterior** | Parametric bootstrap | Exact Gaussian (Cholesky) |
| **One-step-ahead RMSE** | 15.10 | **14.14** |
| **One-step-ahead MAE** | 11.91 | **11.24** |
| **Bias** | −1.45 | −1.17 |
| **Calibration** | — | Near-perfect |
| **Early-season (week 1)** | RMSE 22+ | **RMSE 14.6** |

### What the bias tells us

Both models slightly under-predict late-season games. This is not a flaw — it is the model's static-$o_i$ assumption being violated in a specific, interpretable direction: teams improve during the season. The natural fix is dynamic coefficients $o_i(t)$ evolving as a random walk.

### What this system enables

Ratings are no longer primitive — they are **derived summaries of a posterior**:

$$\theta \sim p(\theta \mid \mathcal{D}) \quad\Rightarrow\quad S(\theta) = (\text{AdjO}, \text{AdjD}, \text{Pace}), \quad \tilde{y}(\theta) = \text{simulated outcome}$$

That means ratings, predictions, and probabilities all flow from the same generative object — and every output carries calibrated uncertainty.